<a href="https://colab.research.google.com/github/Mmbsaksd/transformers/blob/main/Annotated_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Prelims**

In [ ]:
# # Uncomment for colab
# #
!pip install -q  GPUtil
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 83.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 79.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import os
from os.path import exists
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax, pad
import math
import copy
import time
from torch.optim.lr_scheduler import LambdaLR
import pandas as pd
import altair as alt
from torch.utils.data import DataLoader, Dataset
import spacy
import GPUtil
import warnings
from torch.utils.data.distributed import DistributedSampler
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from collections import Counter
from datasets import load_dataset

warnings.filterwarnings("ignore")
RUN_EXAMPLES = True

In [ ]:
def is_interactive_notebook():
    return __name__ == "__main__"


def show_example(fn, args=[]):
    if __name__ == "__main__" and RUN_EXAMPLES:
        return fn(*args)


def execute_example(fn, args=[]):
    if __name__ == "__main__" and RUN_EXAMPLES:
        fn(*args)


class DummyOptimizer(torch.optim.Optimizer):
    def __init__(self):
        self.param_groups = [{"lr": 0}]
        None

    def step(self):
        None

    def zero_grad(self, set_to_none=False):
        None


class DummyScheduler:
    def step(self):
        None

### Background

Earlier models such as Extended Neural GPU, ByteNet, and ConvS2S also tried to reduce step-by-step computation. They used convolutional neural networks (CNNs) to process many positions in the input at the same time.

However, these models had a problem when two words were far apart in a sentence. The farther apart the words were, the more computation was needed to connect them. ConvS2S needed more computation as the distance increased, while ByteNet increased more slowly but still needed additional computation.

The Transformer solves this problem using **self-attention**. Self-attention allows any word to directly look at other words in the sequence, even when they are far apart. This makes it easier for the model to learn relationships between distant words.

Self-attention was already used in some NLP tasks before the Transformer. However, the Transformer was the first model to use self-attention as the main mechanism for processing both the input and output, without using RNNs or CNNs.

In simple terms:

**Older models:** Farther words → more computation to connect them.

**Transformer:** Farther words → can still directly connect through self-attention.


## **Part 1: Model Architecture**

## Model Architecture

Most sequence-to-sequence models use two main parts: an **Encoder** and a **Decoder**.

The **Encoder** takes the input sequence and converts it into useful numerical representations. For example, the input words are converted into vectors, and the Encoder uses the relationships between the words to create better representations.

The **Decoder** uses these representations to generate the output sequence. It generates the output **one token at a time**. When generating the next token, it can use the tokens it has already generated.

The Transformer follows this Encoder-Decoder structure, but instead of using RNNs or CNNs to process the sequence, it mainly uses **attention mechanisms**.

The Encoder is made up of several identical layers. Each layer contains two main parts:

1. **Multi-Head Self-Attention** – allows each word to look at other words in the input and understand their relationships.
2. **Feed-Forward Network** – processes the information from the attention layer further.

The Decoder is also made up of several identical layers. Each layer contains three main parts:

1. **Masked Multi-Head Self-Attention** – allows the decoder to look at previously generated words, but prevents it from looking at future words.
2. **Encoder-Decoder Attention** – allows the decoder to look at the information produced by the Encoder.
3. **Feed-Forward Network** – processes the information further.

The Transformer also uses **residual connections and normalization** around these parts to help the network train effectively.

In simple terms:

**Input → Encoder → Information → Decoder → Output**

The main idea is that the **Encoder understands the input**, and the **Decoder uses that information to generate the output one token at a time**.


In [ ]:
class EncoderDecoder(nn.Module):
    """
    A standard Encoder-Decoder architecture. Base for this and many
    other models.
    """

    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def forward(self, src, tgt, src_mask, tgt_mask):
        "Take in and process masked src and target sequences."
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)

    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

## **Encoder and Decoder Stacks**
### Encoder
The encoder is composed of a stack of  N=6  identical layers

In [ ]:
def clones(module, N):
    "Produce N identical layers."
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [ ]:
class Encoder(nn.Module):
    "Core encoder is a stack of N layers"

    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, mask):
        "Pass the input (and mask) through each layer in turn."
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

### Residual Connections and Layer Normalization

Each sub-layer in the Transformer uses a **residual connection** followed by **Layer Normalization**.

A residual connection adds the original input to the output of the sub-layer:

```text
Input → Sub-layer → Output
  └──────────────────┘
          +
```

In simple terms:

> **Residual connection = keep the original information and add it to the new information.**

After adding them, **Layer Normalization** is applied to keep the values in a stable range and help the model train better.

The Encoder has two sub-layers:

1. Multi-Head Self-Attention
2. Feed-Forward Network

Both use the same pattern:

```text
Input
  ↓
Sub-layer
  ↓
Add original input
  ↓
Layer Normalization
  ↓
Next layer
```

So, the main idea is:

> **Residual connection helps preserve the original information, while Layer Normalization helps the model train more smoothly.**


In [ ]:
class LayerNorm(nn.Module):
    "Construct a layernorm module (See citation for details)."

    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

### Residual Connection, Dropout, and `d_model`

Each Transformer sub-layer follows this basic flow:

```text
Input
  ↓
Sub-layer
  ↓
Dropout
  ↓
Add original Input
  ↓
Layer Normalization
  ↓
Output
```

The calculation is:

```text
Output = LayerNorm(Input + Dropout(Sublayer(Input)))
```

* **Residual connection:** The original input is added back to the sub-layer output. This helps preserve the original information.
* **Dropout:** Some values from the sub-layer output are randomly dropped during training. This helps reduce overfitting.
* **Layer Normalization:** The result is normalized to make training more stable.
* **`d_model = 512`:** Every major representation in the Transformer has 512 features. This is important because the original input and the sub-layer output need compatible dimensions for the residual addition.

For example:

```text
Input:          512 features
Sub-layer:      512 features
       ↓
    Add them
       ↓
Output:         512 features
```

So the main idea is:

> **Process the input → apply dropout → add the original input → normalize the result.**


In [ ]:
class SublayerConnection(nn.Module):
    """
    A residual connection followed by a layer norm.
    Note for code simplicity the norm is first as opposed to last.
    """

    def __init__(self, size, dropout):
        super(SublayerConnection, self).__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        "Apply residual connection to any sublayer with the same size."
        return x + self.dropout(sublayer(self.norm(x)))

### Encoder Layer

Each Encoder layer has **two sub-layers**:

1. **Multi-Head Self-Attention** – allows each token to look at other tokens in the input sequence and understand their relationships.

2. **Position-wise Feed-Forward Network** – further processes the representation of each token independently.

In simple terms:

> **Self-Attention understands relationships between tokens, and the Feed-Forward Network processes each token's information further.**


In [ ]:
class EncoderLayer(nn.Module):
    "Encoder is made up of self-attn and feed forward (defined below)"

    def __init__(self, size, self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size

    def forward(self, x, mask):
        "Follow Figure 1 (left) for connections."
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        return self.sublayer[1](x, self.feed_forward)

### Decoder

The decoder is also composed of a stack of $N=6$ identical layers.

In [ ]:
class Decoder(nn.Module):
    "Generic N layer decoder with masking."

    def __init__(self, layer, N):
        super(Decoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)

### Decoder Layer

Each **Decoder layer has three sub-layers**:

1. **Masked Multi-Head Self-Attention** – allows the Decoder to look at previously generated target tokens, but not future tokens.

2. **Encoder-Decoder Multi-Head Attention** – allows the Decoder to look at the **Encoder's output** and use the relevant information from the input sequence.

3. **Position-wise Feed-Forward Network** – further processes the information for each token.

Each sub-layer uses the same:

```text
Sub-layer
   ↓
Dropout
   ↓
Residual Connection
   ↓
Layer Normalization
```

So the Decoder layer is:

```text
Target
  ↓
Masked Self-Attention
  ↓
Encoder-Decoder Attention
  ↓
Feed-Forward Network
  ↓
Output
```

> **In simple terms: The Decoder first looks at the previous target tokens, then looks at the Encoder's output to find useful input information, and finally processes that information further.**


In [ ]:
class DecoderLayer(nn.Module):
    "Decoder is made of self-attn, src-attn, and feed forward (defined below)"

    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super(DecoderLayer, self).__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)

    def forward(self, x, memory, src_mask, tgt_mask):
        "Follow Figure 1 (right) for connections."
        m = memory
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        return self.sublayer[2](x, self.feed_forward)


### Decoder Self-Attention Masking

In the Decoder's **self-attention**, we prevent a position from looking at **future positions**.

For example, when predicting the next token:

```text
I → love → you
```

When predicting **"you"**, the Decoder can use:

```text
I ✓
love ✓
you ✗  ← future/current answer is hidden
```

This is done using a **mask**.

The target sequence is also shifted by one position, so the Decoder receives the **previous tokens** as input and predicts the next token.

```text
Decoder input:   <START>   I       love
                   ↓        ↓        ↓
Predicts:           I      love      you
```

Therefore:

> **The Decoder can only use tokens that are already known when predicting the next token. It cannot look at future tokens.**

In simple terms:

**Masking + shifting the target by one position = prevents the Decoder from cheating by seeing the correct future answer.**


In [ ]:
def subsequent_mask(size):
    "Mask out subsequent positions."
    attn_shape = (1, size, size)
    subsequent_mask = torch.triu(torch.ones(attn_shape), diagonal=1).type(
        torch.bool
    )
    return subsequent_mask == 0

In [ ]:
def example_mask():
    LS_data = pd.concat(
        [
            pd.DataFrame(
                {
                    "Subsequent Mask": subsequent_mask(20)[0][x, y].flatten(),
                    "Window": y,
                    "Masking": x,
                }
            )
            for y in range(20)
            for x in range(20)
        ]
    )

    return (
        alt.Chart(LS_data)
        .mark_rect()
        .properties(height=250, width=250)
        .encode(
            alt.X("Window:O"),
            alt.Y("Masking:O"),
            alt.Color("Subsequent Mask:Q", scale=alt.Scale(scheme="viridis")),
        )
        .interactive()
    )


show_example(example_mask)

alt.Chart(...)

## Attention

Attention helps the model decide **which information is important**.

It takes:

* **Query (Q)** – what we are looking for
* **Keys (K)** – information we can compare with the query
* **Values (V)** – the actual information we want to use

The model compares the **Query with every Key** to find how relevant each one is.

It then gives higher weight to more relevant Values and combines them to produce the final output.

```text
Query + Keys
     ↓
Calculate how relevant they are
     ↓
Attention weights
     ↓
Weighted combination of Values
     ↓
Output
```

### Scaled Dot-Product Attention

The Transformer uses a specific type of attention called **Scaled Dot-Product Attention**.

The basic steps are:

```text
1. Compare Query with every Key using dot product
2. Divide the scores by √dₖ
3. Apply Softmax to convert scores into attention weights
4. Use these weights to combine the Values
5. Get the final attention output
```

In simple terms:

> **The Query asks "What am I looking for?", the Keys help find which information is relevant, and the Values provide the actual information.**

The formula is:

```text
Attention(Q, K, V) = softmax(QKᵀ / √dₖ)V
```

Where:

* `Q` = Queries
* `K` = Keys
* `V` = Values
* `dₖ` = dimension (number of features) of the Keys
* `√dₖ` = scaling factor used to keep the attention scores stable.


## Attention Using Matrices

In practice, we calculate attention for **all tokens at the same time** instead of calculating one token at a time.

The Query vectors are stored together in a matrix **Q**. Similarly, all Key vectors are stored in **K**, and all Value vectors are stored in **V**.

For example, if we have **7 tokens** and each vector has **4 features**:

```text
Q = 7 × 4
K = 7 × 4
V = 7 × 4
```

Each row represents one token.

### Step 1: Compare every Query with every Key

We calculate:

```text
Q × Kᵀ
```

```text
(7 × 4) × (4 × 7)
        ↓
      7 × 7
```

The resulting **7 × 7 matrix** contains the relationship/compatibility score between every Query and every Key.

```text
Rows    → Queries
Columns → Keys
```

So, for example:

```text
Row 1 → Query of token 1 compared with all 7 Keys
Row 2 → Query of token 2 compared with all 7 Keys
...
Row 7 → Query of token 7 compared with all 7 Keys
```

A **higher score means stronger relationship** between that Query and Key.

### Step 2: Scale the scores

We divide the scores by:

```text
√dₖ
```

This keeps the values at a reasonable scale.

### Step 3: Apply Softmax

Softmax converts the scores into **attention weights**.

```text
Raw scores
    ↓
Softmax
    ↓
Attention weights
```

The weights tell us:

> **How much attention should each Query give to each Key?**

The formula for the complete process is:

```text
Attention(Q, K, V) = softmax(QKᵀ / √dₖ)V
```

### Simple flow

```text
Q, K, V
  ↓
Q × Kᵀ
  ↓
Relationship scores
  ↓
÷ √dₖ
  ↓
Softmax
  ↓
Attention weights
  ↓
× V
  ↓
Attention Output
```

> **In simple words: We compare every Query with every Key, convert the comparison scores into attention weights, and use those weights to combine the Values and produce the final attention output.**


# The Annotated Transformer — Simple Notes

## 1. Background

Older models such as Extended Neural GPU, ByteNet, and ConvS2S also tried to reduce step-by-step computation. They used CNNs to process many positions at the same time.

However, when two words were far apart, these models needed more computation to connect them. This made it harder to learn relationships between distant words.

The Transformer uses **self-attention**, which allows words to directly connect with other words, even when they are far apart.

**Self-attention = a word can look at other words in the same sequence to understand their relationship.**

The Transformer was the first sequence-to-sequence model that relied completely on self-attention instead of RNNs or CNNs.

---

# 2. Model Architecture

Most sequence-to-sequence models have two main parts:

```text
Input
  ↓
Encoder
  ↓
Information
  ↓
Decoder
  ↓
Output
```

### Encoder

The Encoder takes the input tokens and converts them into numerical representations.

```text
Input tokens → Encoder → Encoder representations
```

For example:

```text
"I love you"
     ↓
  Encoder
     ↓
z₁, z₂, z₃
```

Each `z` is a vector containing information about the corresponding input position and its context.

### Decoder

The Decoder uses the Encoder's information to generate the output.

It generates the output **one token at a time**.

```text
Encoder information
       ↓
    Decoder
       ↓
token 1 → token 2 → token 3 → ...
```

The Decoder is **autoregressive**, which means:

> When generating the next token, it uses the tokens that it has already generated.

---

# 3. `EncoderDecoder` Class

The `EncoderDecoder` class connects the main parts of the Transformer:

```text
Source
  ↓
Source Embedding
  ↓
Encoder
  ↓
Memory
  ↓
Decoder
  ↓
Generator
  ↓
Output
```

### Main components

```python
self.encoder
```

Processes the source/input.

```python
self.decoder
```

Generates the target/output using the encoder information.

```python
self.src_embed
```

Converts source token IDs into vectors.

```python
self.tgt_embed
```

Converts target token IDs into vectors.

```python
self.generator
```

Converts the decoder's output into scores for the vocabulary.

---

# 4. `forward()`

```python
return self.decode(
    self.encode(src, src_mask),
    src_mask,
    tgt,
    tgt_mask
)
```

The main flow is:

```text
src
 ↓
encode()
 ↓
Encoder output
 ↓
decode()
 ↓
Decoder output
```

So:

> **First encode the source, then use that information to decode the target.**

---

# 5. `encode()`

```python
def encode(self, src, src_mask):
    return self.encoder(self.src_embed(src), src_mask)
```

The source tokens first go through the embedding:

```text
src
 ↓
src_embed
 ↓
vectors
 ↓
Encoder
```

`src_mask` tells the Encoder which source positions should be ignored, mainly **padding positions**.

The Encoder is not using `src_mask` to hide future words.

---

# 6. `decode()`

```python
def decode(self, memory, src_mask, tgt, tgt_mask):
    return self.decoder(
        self.tgt_embed(tgt),
        memory,
        src_mask,
        tgt_mask
    )
```

The target tokens are converted into vectors:

```text
tgt
 ↓
tgt_embed
 ↓
target vectors
```

The Decoder then uses:

* target vectors
* Encoder output (`memory`)
* `src_mask`
* `tgt_mask`

The target mask prevents the Decoder from seeing future tokens and also handles target padding.

---

# 7. Generator

```python
self.proj = nn.Linear(d_model, vocab)
```

The Decoder produces a vector of size `d_model`.

The Generator converts that vector into scores for every token in the vocabulary.

```text
Decoder output
      ↓
Linear layer
      ↓
Vocabulary scores
      ↓
log_softmax
      ↓
Score/probability for each token
```

If the vocabulary contains 30,000 tokens, the Generator produces a score for each of those 30,000 possible tokens.

```python
return log_softmax(self.proj(x), dim=-1)
```

Here:

```python
self.proj(x)
```

produces the vocabulary scores.

Then:

```python
log_softmax(..., dim=-1)
```

converts those scores into **log-probabilities**.

`dim=-1` means:

> Apply softmax across the last dimension, which is the vocabulary dimension.

---

# 8. Overall Transformer Architecture

The Transformer uses **self-attention and feed-forward networks** instead of RNNs or CNNs.

```text
                 INPUT
                   ↓
             Embedding
                   ↓
              Encoder
        ┌──────────┴──────────┐
        │                     │
   Self-Attention       Feed-Forward
        │                     │
        └────── Encoder ──────┘
                   ↓
                Memory
                   ↓
              Decoder
        ┌──────────┴──────────┐
        │          │          │
   Self-Attention  │   Encoder-Decoder
                   │      Attention
                   │          │
                   └── Feed-Forward
                   ↓
               Generator
                   ↓
                Output
```

---

# 9. Encoder Stack

The Encoder is made of **6 identical layers** in the original Transformer:

```text
Input
 ↓
Encoder Layer 1
 ↓
Encoder Layer 2
 ↓
Encoder Layer 3
 ↓
Encoder Layer 4
 ↓
Encoder Layer 5
 ↓
Encoder Layer 6
 ↓
Encoder output
```

Each layer has the same structure.

---

# 10. `clones()`

```python
def clones(module, N):
    return nn.ModuleList(
        [copy.deepcopy(module) for _ in range(N)]
    )
```

This creates `N` copies of a layer.

For example:

```python
clones(layer, 6)
```

creates:

```text
Layer 1
Layer 2
Layer 3
Layer 4
Layer 5
Layer 6
```

### PyTorch part

```python
copy.deepcopy(module)
```

creates a separate copy of the module.

```python
nn.ModuleList(...)
```

tells PyTorch:

> These are multiple neural-network modules that belong to my model.

Each layer has the **same structure**, but each copy has its **own learnable parameters**.

---

# 11. Encoder Layer — Two Sub-Layers

Each Encoder layer has **two sub-layers**:

### Sub-layer 1

**Multi-Head Self-Attention**

It allows each token to look at other tokens in the input.

```text
Input
 ↓
Self-Attention
 ↓
Context-aware information
```

### Sub-layer 2

**Position-wise Feed-Forward Network**

It processes each position's representation further.

```text
Attention output
 ↓
Feed-Forward Network
 ↓
Processed representation
```

So:

```text
Encoder Layer

Input
 ↓
Multi-Head Self-Attention
 ↓
Feed-Forward Network
 ↓
Output
```

---

# 12. Residual Connection

Each sub-layer uses a residual connection.

The basic flow is:

```text
Input
  ↓
Sub-layer
  ↓
Dropout
  ↓
+ original Input
  ↓
LayerNorm
  ↓
Output
```

The formula is:

```text
LayerNorm(Input + Dropout(Sublayer(Input)))
```

### Residual connection

The original input is added back to the sub-layer output.

> **It helps preserve the original information and makes deep networks easier to train.**

---

# 13. Layer Normalization

LayerNorm normalizes the values in each token's feature vector.

```text
Input
 ↓
Calculate mean
 ↓
Calculate standard deviation
 ↓
Normalize
 ↓
Learnable scale
 ↓
Learnable shift
 ↓
Output
```

The code:

```python
self.a_2 = nn.Parameter(torch.ones(features))
```

creates a **learnable scale**.

```python
self.b_2 = nn.Parameter(torch.zeros(features))
```

creates a **learnable shift**.

```python
mean = x.mean(-1, keepdim=True)
```

calculates the mean across the **last dimension**, which is the feature dimension.

```python
std = x.std(-1, keepdim=True)
```

calculates the standard deviation across the feature dimension.

```python
(x - mean) / (std + self.eps)
```

normalizes the values.

Finally:

```python
self.a_2 * normalized + self.b_2
```

allows the model to learn the best scale and shift.

---

# 14. Dropout

Dropout is applied to the sub-layer output **before** the residual addition.

```text
Sub-layer
   ↓
Dropout
   ↓
+ original input
   ↓
LayerNorm
```

During training, dropout randomly removes some values.

> **Purpose: reduce overfitting and help the model generalize better.**

---

# 15. `d_model = 512`

The original Transformer uses:

```text
d_model = 512
```

This means each token's main representation has **512 features**.

For example:

```text
"I"
 ↓
[0.2, -0.4, 0.7, ... 512 values ...]
```

The important reason for using the same size everywhere is the residual connection.

We need to add:

```text
Input + Sub-layer output
```

So both should have the same feature size:

```text
512 + 512
```

Therefore:

> **All major sub-layers and embedding layers produce 512-dimensional representations.**

---

# 16. Complete Encoder Layer

Putting everything together:

```text
                  Input
                    │
                    ├─────────────────────┐
                    ↓                     │
          Multi-Head Self-Attention       │
                    ↓                     │
                 Dropout                 │
                    ↓                     │
                    └──────── + ─────────┘
                              ↓
                         LayerNorm
                              ↓
                    ┌─────────┴─────────┐
                    │                   │
                    ↓                   │
             Feed-Forward              │
                    ↓                   │
                 Dropout                │
                    ↓                   │
                    └──────── + ────────┘
                              ↓
                         LayerNorm
                              ↓
                            Output
```

The output then goes to the **next Encoder layer**.

---

# 17. Most Important Things to Remember

```text
Encoder
    ↓
6 identical layers
    ↓
Each layer has:
    ↓
1. Multi-Head Self-Attention
2. Feed-Forward Network
    ↓
Each sub-layer has:
    ↓
Sub-layer
 ↓
Dropout
 ↓
Residual connection
 ↓
LayerNorm
```

And the overall Transformer is:

```text
Input
 ↓
Embedding
 ↓
Encoder × 6
 ↓
Encoder output / Memory
 ↓
Decoder × 6
 ↓
Linear + Softmax
 ↓
Output
```

### In one sentence:

> **The Transformer takes the input, creates embeddings, processes them through 6 Encoder layers using self-attention and feed-forward networks, and then the Decoder uses this information to generate the output one token at a time.**
